# RapidFire AI RAG Experiment - Documentation Q&A Chatbot

This notebook runs a RAG-based evaluation pipeline on the RapidFire AI documentation using a fixed chunking configuration.

## Load API Key

In [1]:
from pathlib import Path
import os

# Get API Key
TRITON_API_KEY = Path("~/api-key.txt").expanduser().read_text(encoding="utf-8").splitlines()[0].strip()

# Set environment variables
os.environ.setdefault("OPENAI_API_KEY", TRITON_API_KEY)
os.environ.setdefault("JUDGE_BASE_URL", "https://tritonai-api.ucsd.edu/v1")
os.environ.setdefault("JUDGE_MODEL", "api-gpt-oss-120b")

'api-gpt-oss-120b'

## Import Required Libraries

In [2]:
# RapidFireAI Imports
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
from rapidfireai import Experiment

# Standard library imports
import re
import json
from typing import List as listtype, Dict, Any, Optional
from pathlib import Path

# Data and ML imports
import pandas as pd
from datasets import Dataset

# LangChain imports
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_openai import OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_huggingface import HuggingFaceEmbeddings

# Import evaluation functions
from project1_eval import call_judge, f1_at_k, precision_at_k, recall_at_k, to_spans
from rapidfire_integration_example import sample_compute_metrics_fn, sample_accumulate_metrics_fn

INFO 05-07 10:55:38 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 05-07 10:55:38 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.


## Load Dataset

Load the validation set golden Q&A pairs and experiment.json file.

In [ ]:
# Load input JSON file
input_file = "validation-set-golden-qa-pairs.json"
output_file = "experiment_output2.json"

with open(input_file, "r") as f:
    data = json.load(f)

# Build dataset rows
rows = [
    {
        "query_id": int(entry["question_id"]),          
        "query": str(entry["question"]),
        "reference_answer": str(entry.get("reference_answer", "")), 
        "source_evidence": entry.get("source_evidence", []),
    }
    for entry in data
]

dataset = Dataset.from_list(rows)
print(f"Loaded {len(dataset)} examples from {input_file}")
print(f"Dataset preview:")
print(dataset[0])

Loaded 20 examples from golden-qa.json
Dataset preview:
{'query_id': 1, 'query': 'What are the two things you need to create a multi-config specification in RapidFire AI?', 'reference_answer': 'To create a multi-config specification in RapidFire AI, you need to have two things, knob set generators (to generate knob values) and config group generators (to generate groups of full configs using set-valued knobs)', 'source_evidence': [{'file': 'configs.rst', 'lines': [19, 21]}]}


## Create Experiment

In [ ]:
experiment = Experiment(experiment_name="experiment_hybrid_8", mode="evals")

An experiment with the same name already exists. Created a new experiment 'experiment_hybrid_golden_pair_10' with Experiment ID: 60 at /home/ostran/rapidfireai/rapidfire_experiments/experiment_hybrid_golden_pair_10
Created directory: /home/ostran/rapidfireai/logs/experiment_hybrid_golden_pair_10


## Define RAG Configuration

Configure the LangChain RAG pipeline with chunk sizes 128 and 512.

In [5]:
# Chunking parameters - test both 128 and 512
CHUNK_SIZES = [512]
CHUNK_OVERLAP = 32
SEARCH_KS = [15]
RERANK_TOP_N = [2]
batch_size = 32

# Shared source docs loaded once, then split per configuration for FAISS and BM25.
document_loader = DirectoryLoader(
    path="sourcedocs/sourcedocs/",
    glob="**/*.rst",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    sample_seed=1337,
)
all_docs = document_loader.load()


def build_rag_for_chunking(chunk_size: int, chunk_overlap: int) -> RFLangChainRagSpec:
    """Build a RAG configuration for a given chunk size and overlap."""
    return RFLangChainRagSpec(
        document_loader=DirectoryLoader(
            path="sourcedocs/sourcedocs/",
            glob="**/*.rst",
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"},
            sample_seed=1337,
        ),
        text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name="gpt2",
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            add_start_index=True,
        ),
        embedding_cfg=
        {
            "class": OpenAIEmbeddings,
            "model": "api-tgpt-embeddings",
            "api_key": TRITON_API_KEY,
            "base_url": "https://tritonai-api.ucsd.edu",
            "check_embedding_ctx_length": False,
        },
        vector_store_cfg={"type": "faiss", "batch_size": batch_size},
        search_cfg={"type": "similarity", "k": 15},
        enable_gpu_search=False,
    )


def build_bm25_for_chunking(chunk_size: int, chunk_overlap: int) -> BM25Retriever:
    """Build a BM25 retriever over the same chunked docs used by FAISS."""
    chunk_text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="gpt2",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )
    chunked_docs = chunk_text_splitter.split_documents(all_docs)
    bm25_retriever = BM25Retriever.from_documents(chunked_docs)
    bm25_retriever.k = 10
    return bm25_retriever

bm25_retriever_by_chunk_size = {
    chunk_size: build_bm25_for_chunking(chunk_size=chunk_size, chunk_overlap=CHUNK_OVERLAP)
    for chunk_size in CHUNK_SIZES
}

def get_bm25_retriever_for_rag(rag: RFLangChainRagSpec):
    """Return the BM25 retriever matching the RAG chunk size, falling back safely if needed."""
    splitter = getattr(rag, "text_splitter", None)
    chunk_size = getattr(splitter, "chunk_size", getattr(splitter, "_chunk_size", None))
    return bm25_retriever_by_chunk_size.get(chunk_size, next(iter(bm25_retriever_by_chunk_size.values())))


cross_encoder = HuggingFaceCrossEncoder(
    #model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_name="BAAI/bge-reranker-v2-m3",
    model_kwargs={"device": "cpu"},
)

# create_hybrid_retriever now accepts a per-call `rerank_top_n` so the grid can vary it
def create_hybrid_retriever(faiss_retriever, bm25_retriever, weights=(0.3, 0.7), rerank_top_n=2):
    hybrid = EnsembleRetriever(
        retrievers=[faiss_retriever, bm25_retriever],
        weights=list(weights),
    )
    reranker = CrossEncoderReranker(model=cross_encoder, top_n=rerank_top_n)
    return ContextualCompressionRetriever(
        base_compressor=reranker,
        base_retriever=hybrid,
    )

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

## Define Instructions and Helper Functions

In [6]:
INSTRUCTIONS = """You are a precise technical assistant for the RapidFire AI documentation.
You will be given a user question and relevant context chunks retrieved from the RapidFire AI docs.

Rules:
- Answer using ONLY information present in the provided context. Do not use outside knowledge.
- Be specific and complete — include parameter names, types, defaults, and exact values when present.
- For procedural questions, list the steps in order.
- For comparative questions, clearly distinguish between the two things being compared.
- For factual/lookup questions, give the exact answer directly.
- If the context does not contain enough information to answer, say: "The provided context does not contain enough information to answer this question."
- Do not add caveats, filler phrases, or unnecessary preamble. Get to the answer immediately.
- Stay within 2000 tokens total context budget.

Respond with your answer only. No reasoning prefix needed.
Question: "What are the two main execution functions provided by the Experiment class for launching workflows?"
Answer: "The two main execution functions are run_fit() for training/evaluation workflows and run_evals() for LLM evaluation workflows."
Source Evidence: source_evidence": [
    { "file": "experiment.rst", "lines": [69, 75] },
    { "file": "experiment.rst", "lines": [154, 160] }
    ]
"""

# Whitespace-token-aware truncation, safety tokens are a failsafe in case something weird happens with token counting
MAX_TOKENS_PER_QUERY: int = 2000
SAFETY_TOKENS: int = 50

# Get number of white space separated tokens from a text
def count_tokens(text: str) -> int:
    return len(text.split())

# Truncate the context to fit within the limit of tokens
def truncate_tokens(text: str, max_tokens: int) -> str:
    toks = text.split()
    if len(toks) <= max_tokens:
        return text
    return " ".join(toks[:max_tokens])

# Determine a chunk's line numbers
def chunk_to_lines(doc: Document) -> listtype:
    """Convert a chunk's character `start_index` into [start_line, end_line]."""
    try:
        src = doc.metadata.get("source")
        if not src:
            return [1, 1]

        text = Path(src).read_text(encoding="utf-8")
        start_idx = doc.metadata.get("start_index")
        if start_idx is None:
            snippet = (doc.page_content or "").strip()
            if not snippet:
                return [1, 1]
            start_idx = text.find(snippet)
            if start_idx < 0:
                return [1, 1]

        start_line = text[:int(start_idx)].count("\n") + 1
        end_line = start_line + (doc.page_content or "").count("\n")
        if end_line < start_line:
            end_line = start_line
        return [start_line, end_line]
    except Exception:
        return [1, 1]

## Define Preprocessing and Postprocessing Functions

In [7]:
# Initialize output storage
output_rows = []
output_rows_jsonl = Path(output_file + ".rows.jsonl")
if output_rows_jsonl.exists():
    output_rows_jsonl.unlink()

def openai_sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Function to prepare the final inputs given to the generator model using hybrid BM25+FAISS retrieval"""

    faiss_retriever = rag.retriever
    bm25_retriever = get_bm25_retriever_for_rag(rag)
    # allow per-rag rerank_top_n (set when the rag was constructed) falling back to 2
    rerank_top_n = getattr(rag, "rerank_top_n", 2)
    hybrid_retriever = create_hybrid_retriever(faiss_retriever, bm25_retriever, weights=(0.6, 0.4), rerank_top_n=rerank_top_n)

    all_context = []
    for query in batch["query"]:
        hybrid_docs = hybrid_retriever.invoke(query)
        all_context.append(hybrid_docs)

    serialized_context = []
    for docs in all_context:
        context_text = "\n\n".join(
            f"[{Path(doc.metadata.get('source', 'unknown')).name}]\n{doc.page_content}"
            for doc in docs
        )
        serialized_context.append(context_text)

    system_tokens = count_tokens(INSTRUCTIONS or "")
    template_tokens = count_tokens("\nQuestion:\n\nContext:\n\nAnswer:")
    new_serialized = []

    for question, ctx in zip(batch.get("query", []), serialized_context):
        q_tokens = count_tokens(question or "")
        avail = MAX_TOKENS_PER_QUERY - (system_tokens + q_tokens + template_tokens + SAFETY_TOKENS)
        if avail <= 0:
            new_serialized.append("")
        else:
            new_serialized.append(truncate_tokens(ctx, avail))

    serialized_context = new_serialized
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    batch["ground_truth_spans"] = [
        [
            (item["file"], int(item["lines"][0]), int(item["lines"][1]))
            for item in evidence
            if item.get("file") and item.get("lines") and len(item["lines"]) >= 2
        ]
        for evidence in batch.get("source_evidence", [])
    ]

    per_doc_lines = [[chunk_to_lines(doc) for doc in docs] for docs in all_context]

    return {
        "prompts": [
            [
                {"role": "system", "content": INSTRUCTIONS},
                {"role": "user", "content": f"\nQuestion:\n{question}\n\nContext:\n{context}\n\nAnswer:"},
            ]
            for question, context in zip(batch["query"], serialized_context)
        ],
        "serialized_context": serialized_context,
        "retrieved_context": serialized_context,
        "sources": [
            [
                {"file": Path(doc.metadata["source"]).name, "lines": lines}
                for doc, lines in zip(docs, doc_lines)
            ]
            for docs, doc_lines in zip(all_context, per_doc_lines)
        ],
        "retrieved_spans": [
            [
                (Path(doc.metadata["source"]).name, lines[0], lines[1])
                for doc, lines in zip(docs, doc_lines)
            ]
            for docs, doc_lines in zip(all_context, per_doc_lines)
        ],
        **batch,
    }

def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Postprocess outputs produced by generator model"""
    batch["answer"] = batch["generated_text"]

    for qid, ans, ctx, srcs in zip(
        batch["query_id"],
        batch["answer"],
        batch["retrieved_context"],
        batch["sources"],
    ):
        row = {
            "question_id": int(qid),
            "answer": ans,
            "retrieved_context": ctx,
            "sources": srcs,
        }
        output_rows.append(row)
        with open(output_rows_jsonl, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    return batch

## Define Generator Configuration

In [8]:
# Create OpenAI generator configs for each config
openai_configs = []
for chunk_size in CHUNK_SIZES:
    for search_k in SEARCH_KS:
        for rerank_top_n in RERANK_TOP_N:
            # Build rag spec for this chunking and then override search k and attach rerank value
            rag = build_rag_for_chunking(chunk_size=chunk_size, chunk_overlap=CHUNK_OVERLAP)
            # override the search k used by this RAG spec
            rag.search_cfg = {"type": "similarity", "k": int(search_k)}
            # attach rerank_top_n so downstream code can create a reranker per-config
            setattr(rag, "rerank_top_n", int(rerank_top_n))

            openai_configs.append(
                RFOpenAIAPIModelConfig(
                    client_config={
                        "api_key": TRITON_API_KEY,
                        "base_url": "https://tritonai-api.ucsd.edu",
                        "max_retries": 2,
                    },
                    model_config={
                        "model": "api-mistral-small-3.2-2506",
                        "max_completion_tokens": 2048,
                        "temperature": 0.8,
                    },
                    rpm_limit=120,
                    tpm_limit=1_000_000,
                    rag=rag,
                    prompt_manager=None,
                )
            )

print(f"Built {len(openai_configs)} generator configs")

Built 1 generator configs


## Run Evaluation

In [9]:
# Launch evals one config at a time, but keep 4-way sharding inside each run
results = []
for config_index, openai_config in enumerate(openai_configs, start=1):
    rag = getattr(openai_config, "rag", None)
    rerank_top_n = getattr(rag, "rerank_top_n", None)
    model_config = getattr(openai_config, "model_config", {})
    model_name = model_config.get("model") if isinstance(model_config, dict) else getattr(model_config, "model", None)

    print(
        f"Running config {config_index}/{len(openai_configs)}: "
        f"model_name={model_name}, rerank_top_n={rerank_top_n}"
    )

    single_config_set = {
        "openai_config": List([openai_config]),
        "batch_size": batch_size,
        "preprocess_fn": openai_sample_preprocess_fn,
        "postprocess_fn": sample_postprocess_fn,
        "compute_metrics_fn": sample_compute_metrics_fn,
        "accumulate_metrics_fn": sample_accumulate_metrics_fn,
    }
    single_config_group = RFGridSearch(single_config_set)

    run_result = experiment.run_evals(
        config_group=single_config_group,
        dataset=dataset,
        num_shards=4,
        num_actors=4,
        seed=42,
    )
    results.append(run_result)

print("Evaluation completed!")

Running config 1/1: model_name=api-mistral-small-3.2-2506, rerank_top_n=2


=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device,Vector Store
1,Complete,29.5s,CPU,FAISS



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,text_splitter,chunk_size,chunk_overlap,embedding_cfg.base_url,embedding_cfg.check_embedding_ctx_length,embedding_cfg.class,embedding_cfg.model,vector_store_cfg.batch_size,vector_store_cfg.type,search_cfg.k,search_cfg.type,model_config,Completeness_normalized,Correctness_pass_rate,F1_at_5,Faithfulness_pass_rate,Generation_Score_3_released,Judge Failures,Precision_at_5,Processing Time,Recall_at_5,Retrieval Score,Samples Per Second,Samples Processed,Throughput,Total
1,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,512,32,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,15,similarity,"max_completion_tokens=2048, temperature=0.8","0.7600 [0.7600, 0.7600]","0.8500 [0.8500, 0.8500]","0.4802 [0.4802, 0.4802]","0.8000 [0.8000, 0.8000]","0.8033 [0.8033, 0.8033]",0.0000,"0.4500 [0.4500, 0.4500]",1582.08 seconds,"0.6200 [0.6200, 0.6200]","0.5167 [0.5167, 0.5167]",0.01,20,0.0/s,20


Evaluation completed!


## Save Output

In [10]:
# Write final output JSON
output_rows.sort(key=lambda x: x["question_id"])

if output_rows_jsonl.exists():
    with open(output_rows_jsonl, "r", encoding="utf-8") as f:
        output_rows = [json.loads(line) for line in f if line.strip()]
    output_rows.sort(key=lambda x: x["question_id"])

with open(output_file, "w") as f:
    json.dump(output_rows, f, indent=2)

print(f"Output saved to {output_file}")
print(f"Total examples processed: {len(output_rows)}")

Output saved to experiment_output2.json
Total examples processed: 20


## End Experiment

In [11]:
experiment.end()
print("Experiment ended.")

Experiment experiment_hybrid_golden_pair_10 ended
Experiment ended.


## View Experiment Logs

In [12]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")

📄 Log File: /home/ostran/rapidfireai/logs/experiment_hybrid_golden_pair_10/rapidfire.log

Last 30 lines of rapidfire.log:
2026-05-07 11:09:24 | Controller | INFO | controller.py:1299 | [experiment_hybrid_golden_pair_10:Controller] Pipeline 1 completed shard 1 (1 batches, 377.54s)
2026-05-07 11:09:24 | Controller | INFO | controller.py:1422 | [experiment_hybrid_golden_pair_10:Controller] Scheduling pipeline 1 (Pipeline 1) on actor 0 for shard 2 (1 batches)
2026-05-07 11:09:24 | QueryProcessingActor-0 | INFO | query_actor.py:169 | [experiment_hybrid_golden_pair_10:QueryProcessingActor-0] Reusing existing inference engine (config hash: 6ed0c3b1)
2026-05-07 11:09:24 | QueryProcessingActor-0 | INFO | query_actor.py:198 | [experiment_hybrid_golden_pair_10:QueryProcessingActor-0] Recreated embedding function: OpenAIEmbeddings
2026-05-07 11:09:24 | QueryProcessingActor-0 | INFO | query_actor.py:211 | [experiment_hybrid_golden_pair_10:QueryProcessingActor-0] Using CPU-based FAISS for retrieval 